# Lesson 03a — Embeddings & Chunking Strategies
**Domain:** Scientific Literature  
**Dataset:** `gfissore/arxiv-abstracts-2021` (HuggingFace, 2000 abstracts)  
**Time estimate:** ~2 h

## Learning objectives
- `sentence-transformers`: `all-MiniLM-L6-v2` (local, no API)
- OpenAI `text-embedding-3-small` (API alternative)
- Chunking: fixed-size, sentence, semantic / paragraph
- Cosine similarity: manual implementation + numpy

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from llm_checker import check, hint, evaluate, progress

local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODEL        = "local-model"

# Load local embedding model (no API needed)
print("Loading all-MiniLM-L6-v2 embedding model…")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded")

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

## Load arXiv abstracts

In [ ]:
from datasets import load_dataset

print("Loading arXiv abstracts…")
ds = load_dataset("gfissore/arxiv-abstracts-2021", split="train").select(range(2000))
print(f"✅ Loaded {len(ds)} abstracts")

# Build a DataFrame for easy access
abstracts_df = pd.DataFrame({
    "title":    [row.get("title", "")    for row in ds],
    "abstract": [row.get("abstract", "") or row.get("text", "") for row in ds],
    "id":       [str(i) for i in range(len(ds))],
})
print(abstracts_df.head(3)[["title", "abstract"]].to_string(max_colwidth=80))

---
## 🔵 EXAMPLE — Ex 03a-1: Embed and retrieve

Embed 50 arXiv abstracts and 10 science questions. For each question,
find the top-1 most similar abstract using cosine similarity.

In [ ]:
# ── Ex 03a-1: Embed and retrieve (EXAMPLE) ────────────────────────
abstracts_50  = abstracts_df["abstract"].tolist()[:50]
titles_50     = abstracts_df["title"].tolist()[:50]

print("Embedding 50 abstracts…")
abs_embeddings = embed_model.encode(abstracts_50, show_progress_bar=True)  # (50, 384)

science_questions = [
    "What is the role of attention mechanisms in neural language models?",
    "How do graph neural networks process molecular structures?",
    "What are the main approaches to protein structure prediction?",
    "How is reinforcement learning applied to robotics?",
    "What is federated learning and how does it preserve privacy?",
    "How do vision transformers differ from convolutional neural networks?",
    "What are the main challenges in natural language understanding?",
    "How is quantum computing applied to optimization problems?",
    "What is contrastive learning in computer vision?",
    "How do diffusion models generate images?",
]

print("Embedding 10 questions…")
q_embeddings = embed_model.encode(science_questions)   # (10, 384)

print(f"\n{'Question':60}  {'Top Match Score':15}  Title")
print("-" * 120)
for i, (q, q_emb) in enumerate(zip(science_questions, q_embeddings)):
    scores   = [cosine_similarity(q_emb, a_emb) for a_emb in abs_embeddings]
    best_idx = int(np.argmax(scores))
    best_score = scores[best_idx]
    print(f"{q[:58]:60}  {best_score:>12.4f}    {titles_50[best_idx][:60]}")

check([
    (abs_embeddings.shape == (50, 384),  "Abstract embeddings shape is (50, 384)"),
    (q_embeddings.shape  == (10, 384),   "Question embeddings shape is (10, 384)"),
    (all(-1 <= cosine_similarity(q_embeddings[0], abs_embeddings[i]) <= 1
         for i in range(50)),            "Cosine similarities in [-1, 1]"),
], exercise_id="03a-1")

---
## 🟡 EXERCISE — Ex 03a-2: Chunking strategy comparison

Split 5 full arXiv papers using 3 methods:
1. Fixed 512 tokens with 50-token overlap
2. Sentence chunking
3. Paragraph chunking

For each: count chunks, avg length, max length.

**Auto-check verifies:**
- Comparison table (3 strategies × 3 metrics) filled
- Fixed chunking gives most uniform chunk sizes
- Comment on which strategy you'd use for scientific papers

In [ ]:
import tiktoken
from langchain.text_splitter import RecursiveCharacterTextSplitter

enc = tiktoken.get_encoding("cl100k_base")

# Use 5 longer abstracts (or concatenate with title for realism)
papers_5 = [
    abstracts_df["title"].iloc[i] + "\n\n" + abstracts_df["abstract"].iloc[i] * 3  # simulate full text
    for i in range(5)
]

def chunk_stats(chunks: list) -> dict:
    lengths = [len(enc.encode(c)) for c in chunks]
    return {
        "n_chunks": len(chunks),
        "avg_tokens": int(np.mean(lengths)) if lengths else 0,
        "max_tokens": int(np.max(lengths))  if lengths else 0,
        "std_tokens": int(np.std(lengths))  if lengths else 0,
    }


def chunking_fixed(text: str, chunk_size: int = 512, overlap: int = 50) -> list:
    # TODO: use RecursiveCharacterTextSplitter with chunk_size and chunk_overlap
    raise NotImplementedError("Implement chunking_fixed()")


def chunking_sentence(text: str) -> list:
    # TODO: split on '. ' or similar sentence boundaries
    # Filter out chunks shorter than 10 characters
    raise NotImplementedError("Implement chunking_sentence()")


def chunking_paragraph(text: str) -> list:
    # TODO: split on '\n\n', re-join short paragraphs (< 50 tokens) with the next one
    raise NotImplementedError("Implement chunking_paragraph()")

In [ ]:
try:
    results_table = []
    for strategy_name, fn in [("fixed_512", chunking_fixed), ("sentence", chunking_sentence), ("paragraph", chunking_paragraph)]:
        all_chunks = []
        for paper in papers_5:
            all_chunks.extend(fn(paper))
        stats = chunk_stats(all_chunks)
        stats["strategy"] = strategy_name
        results_table.append(stats)
        print(f"{strategy_name:15}: {stats['n_chunks']:>5} chunks, "
              f"avg={stats['avg_tokens']:>4} tokens, max={stats['max_tokens']:>5}, std={stats['std_tokens']:>4}")

    fixed_std     = next(r["std_tokens"] for r in results_table if r["strategy"] == "fixed_512")
    sentence_std  = next(r["std_tokens"] for r in results_table if r["strategy"] == "sentence")
    para_std      = next(r["std_tokens"] for r in results_table if r["strategy"] == "paragraph")

    print("\n# Decision: For scientific papers, paragraph chunking preserves argument structure best.")

    check([
        (len(results_table) == 3,       "All 3 strategies computed"),
        (fixed_std <= sentence_std,     "Fixed chunking is more uniform than sentence chunking"),
    ], exercise_id="03a-2")

except NotImplementedError as e:
    print(f"⚠️  {e}")

---
## 🔴 CHALLENGE — Ex 03a-3: Recall@3 benchmark

Build 20 (question, expected_abstract_id) pairs from arXiv data.
Compute Recall@3 for `all-MiniLM-L6-v2`.
Optionally compare to OpenAI `text-embedding-3-small`.

**Recall@3** = count(expected_id in top_3_results) / 20

In [ ]:
# Build 20 ground-truth pairs
# Strategy: for each abstract, generate a question that should retrieve it
ground_truth_pairs = []

def llm(prompt, max_tokens=60):
    r = local_client.chat.completions.create(
        model=MODEL, messages=[{"role":"user","content":prompt}],
        max_tokens=max_tokens, temperature=0.5
    )
    return r.choices[0].message.content.strip()

print("Building ground-truth Q&A pairs from abstracts…")
for i in range(20):
    abstract = abstracts_df["abstract"].iloc[i]
    question = llm(f"Write ONE short factual question that can be answered by this abstract (max 15 words):\n{abstract[:400]}")
    ground_truth_pairs.append({"question": question, "expected_id": str(i)})
    if (i+1) % 5 == 0:
        print(f"  Built {i+1}/20 pairs")

print("\nSample pairs:")
for pair in ground_truth_pairs[:3]:
    print(f"  Q: {pair['question']}")
    print(f"  A: abstract {pair['expected_id']}")
    print()

In [ ]:
# Embed all 2000 abstracts for retrieval
print("Embedding 500 abstracts for retrieval benchmark…")
corpus_size  = 500
corpus_texts = abstracts_df["abstract"].tolist()[:corpus_size]
corpus_embs  = embed_model.encode(corpus_texts, show_progress_bar=True, batch_size=64)

# Compute Recall@3
hits_at_3 = 0
for pair in ground_truth_pairs:
    q_emb = embed_model.encode([pair["question"]])[0]
    scores = [cosine_similarity(q_emb, c_emb) for c_emb in corpus_embs]
    top3   = sorted(range(corpus_size), key=lambda i: scores[i], reverse=True)[:3]
    top3_ids = [str(i) for i in top3]
    if pair["expected_id"] in top3_ids:
        hits_at_3 += 1

recall_at_3_minilm = hits_at_3 / len(ground_truth_pairs)
print(f"\nall-MiniLM-L6-v2  →  Recall@3 = {recall_at_3_minilm:.0%}  ({hits_at_3}/{len(ground_truth_pairs)})")
print("Note: OpenAI text-embedding-3-small not always better — local MiniLM is competitive on sci text.")

check([
    (isinstance(recall_at_3_minilm, float),   "Recall@3 computed as float"),
    (0 <= recall_at_3_minilm <= 1,            "Recall@3 in [0, 1]"),
    (len(ground_truth_pairs) == 20,           "20 ground-truth pairs used"),
], exercise_id="03a-3")

In [ ]:
progress()

---
## Readiness checklist before moving to 03b
- [ ] Embeddings generated locally with MiniLM (no API call required)
- [ ] Cosine similarity computed correctly — values in [-1, 1]
- [ ] Chunking strategy comparison table filled
- [ ] Recall@3 measured for at least 1 embedder